# STG_8 — Serializar artefactos para la API

Esta notebook no entrena ni valida un modelo nuevo: reutiliza el modelo ganador y la
metodología ya validados y aprobados en la spec 009 (`STG_6`/`STG_7`). Su único trabajo es
dejar en disco, de forma reproducible, todo lo que la API (spec 010) necesita para predecir
sin recalcular nada pesado en cada request:

1. El modelo LGBM ganador, reentrenado con los mismos hiperparámetros sobre train+holdout
   combinados (para aprovechar toda la información disponible en producción).
2. El modelo de clustering de temas (KMeans, K=20) para poder asignar tema a un título nuevo.
3. Tres tablas "snapshot": el historial acumulado de cada diputado hasta hoy, listo para
   usar en una predicción sin tener que recorrer todo `df_entrenamiento.csv` en cada pedido.

Cada paso se compara contra lo ya documentado en la spec 009 para confirmar que serializar
no cambió el comportamiento del modelo.

In [1]:
# T1 — Carga y reconstrucción de las 394 features del modelo
# Reproduce exactamente la misma preparación de datos que STG_6, para poder reentrenar
# el modelo ganador sobre los mismos datos con los que se validó.

import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

DATA_DIR     = Path("../data")
RANDOM_STATE = 42

# T1 — Cargar dataset y reconstruir las 394 columnas de entrada del modelo
df = pd.read_csv(DATA_DIR / "df_entrenamiento.csv", parse_dates=["fecha_votacion"])
df = df.sort_values("fecha_votacion").reset_index(drop=True)

# Mismas columnas META/TARGET/FEATURES que STG_6 — el modelo espera este orden y estas columnas
META   = ["diputado", "titulo_base", "fecha_votacion", "bloque", "provincia", "tema_label"]
TARGET = "voto"
FEATURES = [c for c in df.columns if c not in META + [TARGET]]

print(f"Filas    : {len(df):,}")
print(f"Features : {len(FEATURES)}")
print(f"Rango de fechas: {df['fecha_votacion'].min().date()} a {df['fecha_votacion'].max().date()}")

assert len(FEATURES) == 394, f"Se esperaban 394 features, hay {len(FEATURES)}"
assert df[FEATURES].isna().sum().sum() == 0, "Hay NaN en las features — no debería, STG_5 ya las rellenó"
assert len(df) == 25082, f"Se esperaban 25.082 filas, hay {len(df)}"
print()
print("T1 PASA: dataset cargado y features reconstruidas igual que en STG_6.")

Filas    : 25,082
Features : 394
Rango de fechas: 1993-12-22 a 2026-05-20

T1 PASA: dataset cargado y features reconstruidas igual que en STG_6.


## T2 — Reentrenar el LGBM ganador sobre todos los datos

Se usan **los mismos hiperparámetros** que ganaron en `STG_6` (no se vuelve a tunear —
`STG_7` ya confirmó que el tuning no mejora sobre el default). La diferencia es que acá se
entrena con **train + holdout combinados**: el holdout de `STG_6` solo servía para medir
qué tan bien generaliza el modelo; ya cumplió su función. Para el modelo que va a predecir
votos reales, conviene que aproveche toda la historia disponible.

Esto no es una evaluación nueva ni leakage: no se mide nada acá, solo se fija el modelo que
va a servir la API. La comparación honesta (F1-macro, matriz de confusión) ya se hizo y quedó
documentada en la spec 009.

In [2]:
# T2 — Reentrenar el LGBM ganador (mismos hiperparametros de STG_6) sobre todo el dataset
from lightgbm import LGBMClassifier

# Reusar el LabelEncoder ya ajustado en STG_6 (mismo orden de clases, no se reajusta)
le_voto = joblib.load(DATA_DIR / "le_voto.joblib")
df["voto_enc"] = le_voto.transform(df[TARGET])

X_full = df[FEATURES].values
y_full = df["voto_enc"].values

# Mismos hiperparametros que el LGBM ganador en STG_6 (sin tuning, STG_7 confirmo que no mejora)
modelo_lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)
modelo_lgbm.fit(X_full, y_full)

joblib.dump(modelo_lgbm, DATA_DIR / "modelo_lgbm.joblib")

print(f"Modelo entrenado sobre {len(df):,} filas (train+holdout combinados).")
print(f"Clases: {list(le_voto.classes_)}")
print(f"Guardado en: {DATA_DIR / 'modelo_lgbm.joblib'}")
print()
print("T2 PASA.")

Modelo entrenado sobre 25,082 filas (train+holdout combinados).
Clases: ['ABSTENCIÓN', 'AFIRMATIVO', 'NEGATIVO']
Guardado en: ..\data\modelo_lgbm.joblib

T2 PASA.


## T3 — Sanity check contra el holdout de `STG_6`

**Importante para la defensa del TP**: este número **no es una métrica de generalización
nueva** y no reemplaza al 0.453 documentado en la spec 009. El modelo de T2 se entrenó con
train **+ holdout combinados**, así que evaluarlo sobre el holdout es evaluarlo sobre datos
que ya vio durante el entrenamiento (no es una prueba honesta de "qué tan bien predice datos
nuevos"). Ese resultado honesto ya se midió correctamente en `STG_6` con un modelo que
**no** había visto el holdout, y es el que hay que citar siempre: **F1-macro holdout = 0.453**.

Lo único que este chequeo verifica es que **serializar no rompió nada**: mismo orden de
columnas, mismo encoder de clases, mismo comportamiento del modelo. Por eso se espera que el
resultado sea igual o mejor al 0.453 (el modelo ahora conoce esas filas); si diera **peor**,
sería señal de un bug real (columnas desalineadas, encoder distinto, etc.) y habría que
frenar antes de seguir.

In [3]:
# T3 - Sanity check: reconstruir el holdout de STG_6 y evaluar el modelo final sobre el
from sklearn.metrics import f1_score

# Mismo split 80/20 por fecha que en STG_6 (recordatorio: aqui es solo para el sanity check,
# el modelo de T2 ya fue entrenado incluyendo estas filas)
corte_idx   = int(len(df) * 0.80)
fecha_corte = df.iloc[corte_idx]["fecha_votacion"]
df_hold     = df.iloc[corte_idx:].copy()

X_hold = df_hold[FEATURES].values
y_hold = df_hold["voto_enc"].values

y_pred_hold   = modelo_lgbm.predict(X_hold)
f1_hold_final = f1_score(y_hold, y_pred_hold, average='macro')

F1_HOLDOUT_STG6 = 0.453  # documentado en specs/009-modelado-prediccion-votos

print(f"Fecha de corte           : {fecha_corte.date()}")
print(f"Filas en el holdout      : {len(df_hold):,}")
print(f"F1-macro (STG_6, honesto, modelo SIN ver el holdout): {F1_HOLDOUT_STG6}")
print(f"F1-macro (este chequeo, modelo CON el holdout ya visto): {f1_hold_final:.3f}")
print()

assert f1_hold_final >= F1_HOLDOUT_STG6, (
    f"El modelo final da PEOR F1-macro ({f1_hold_final:.3f}) que el modelo original "
    f"({F1_HOLDOUT_STG6}) a pesar de haber visto estos datos. Esto indica un bug en la "
    f"serializacion (columnas desalineadas, encoder distinto, hiperparametros mal copiados, etc.)."
)

print("T3 PASA: el refit final no rompio nada. El F1-macro oficial para citar en la defensa")
print(f"sigue siendo el de STG_6 (holdout genuino, sin ver esos datos): {F1_HOLDOUT_STG6}.")

Fecha de corte           : 2025-10-08
Filas en el holdout      : 5,017
F1-macro (STG_6, honesto, modelo SIN ver el holdout): 0.453
F1-macro (este chequeo, modelo CON el holdout ya visto): 0.989

T3 PASA: el refit final no rompio nada. El F1-macro oficial para citar en la defensa
sigue siendo el de STG_6 (holdout genuino, sin ver esos datos): 0.453.


## T4 — Reconstruir y guardar el KMeans de temas

El clustering de `STG_4` (K=20, `random_state=42, n_init=10`) es determinista: mismos datos
+ misma semilla dan siempre el mismo resultado. Por eso alcanza con volver a ajustarlo sobre
los mismos embeddings de `df_features_titulo.csv` y guardarlo — no hay que re-decidir nada
(ni el K, ni qué grupo es `sin_clasificar`). Se verifica que el refit asigna exactamente los
mismos `tema_id` que ya están guardados, como chequeo de que el refit reproduce el original.

Este modelo se usa después en la API solo con `.predict()` (nunca `.fit()`), para asignarle
un tema al título de una ley nueva sin tocar los clusters existentes.

In [4]:
# T4 - Reconstruir el KMeans de temas (K=20) y guardar el modelo + el mapa de etiquetas
from sklearn.cluster import KMeans

df_temas = pd.read_csv(DATA_DIR / "df_features_titulo.csv")
emb_cols = [c for c in df_temas.columns if c.startswith("emb_")]
assert len(emb_cols) == 384

embeddings_temas = df_temas[emb_cols].values

kmeans_temas = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10)
labels_refit = kmeans_temas.fit_predict(embeddings_temas)

# Sanity check: el refit debe asignar el mismo tema_id que ya esta guardado en df_features_titulo.csv
coincidencias = (labels_refit == df_temas["tema_id"].values).mean()
print(f"Coincidencia refit vs tema_id original: {coincidencias*100:.1f}%")
assert coincidencias == 1.0, "El refit del KMeans no reproduce los tema_id originales (revisar orden de filas/columnas)"

joblib.dump(kmeans_temas, DATA_DIR / "kmeans_temas.joblib")

# Mapa tema_id -> tema_label (incluye el grupo 11 = sin_clasificar, ya reflejado en tema_label)
mapa_temas = (
    df_temas[["tema_id", "tema_label"]]
    .drop_duplicates()
    .sort_values("tema_id")
    .set_index("tema_id")["tema_label"]
    .to_dict()
)
mapa_temas = {int(k): v for k, v in mapa_temas.items()}

with open(DATA_DIR / "mapa_temas.json", "w", encoding="utf-8") as f:
    json.dump(mapa_temas, f, ensure_ascii=False, indent=2)

print(f"Guardado: {DATA_DIR / 'kmeans_temas.joblib'}")
print(f"Guardado: {DATA_DIR / 'mapa_temas.json'} ({len(mapa_temas)} temas)")
print()
print("T4 PASA.")

Coincidencia refit vs tema_id original: 100.0%
Guardado: ..\data\kmeans_temas.joblib
Guardado: ..\data\mapa_temas.json (20 temas)

T4 PASA.


## T5 — Snapshot de features del diputado (acumulado total, sin `.shift`)

Se recalculan las tasas historicas del diputado igual que en `STG_5`, pero **sin excluir
ningun voto** (sin `.shift(1)`): para una prediccion sobre una ley nueva no hay ningun voto
"actual" que descartar, asi que corresponde usar **toda** la historia conocida de cada
diputado hasta hoy. Se recalcula desde `df_modelado.csv` (los votos crudos consolidados),
replicando el mismo filtro y las mismas definiciones que `STG_5` (se descartan los votos
AUSENTE porque no son una posicion politica).

El `bloque` y la `provincia` que se guardan son los del **voto mas reciente** de cada
diputado (bastantes diputados cambiaron de bloque durante su historia), para que la
prediccion use su afiliacion actual. La codificacion (`bloque_enc`/`provincia_enc`) usa el
**mismo encoder** ya ajustado en `STG_5` (`encoder_bloque_provincia.joblib`) — no se reajusta,
para que los codigos sean consistentes con los que el modelo aprendio.

In [5]:
# T5 - Snapshot de features del diputado (acumulado total hasta hoy)
CORTE_2023 = pd.Timestamp('2023-12-10')
CORTE_2026 = pd.Timestamp('2026-01-01')

df_votos_raw = pd.read_csv(DATA_DIR / "df_modelado.csv", parse_dates=["fecha_votacion"])
df_votos_raw = df_votos_raw[df_votos_raw['voto'] != 'AUSENTE'].copy()
df_votos_raw['voto'] = df_votos_raw['voto'].replace('ABSTENCION', 'ABSTENCIÓN')
assert len(df_votos_raw) == 25082, "El filtro de AUSENTE no dio el mismo resultado que STG_5"

df_votos_raw['es_afirmativo'] = (df_votos_raw['voto'] == 'AFIRMATIVO').astype(float)

# Igual que STG_5 T3: voto mayoritario del bloque por (titulo, fecha, bloque)
voto_bloque = (
    df_votos_raw.groupby(['titulo_base', 'fecha_votacion', 'bloque'])['voto']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={'voto': 'voto_bloque'})
)
df_votos_raw = df_votos_raw.merge(voto_bloque, on=['titulo_base', 'fecha_votacion', 'bloque'], how='left')
df_votos_raw['alineado_bloque'] = (df_votos_raw['voto'] == df_votos_raw['voto_bloque']).astype(float)

# Acumulado TOTAL por diputado (sin shift: no hay voto "actual" que excluir)
snapshot_diputado = df_votos_raw.groupby('diputado').agg(
    tasa_afirmativo_hist=('es_afirmativo', 'mean'),
    tasa_alineacion_bloque_hist=('alineado_bloque', 'mean'),
    n_votos_hist=('es_afirmativo', 'size'),
)

# Tasas con ventana temporal: solo sobre los votos dentro de la ventana; NaN si no voto en esa ventana
for col_out, corte in [('tasa_afirmativo_desde_2023', CORTE_2023), ('tasa_afirmativo_2026', CORTE_2026)]:
    ventana = df_votos_raw[df_votos_raw['fecha_votacion'] >= corte]
    snapshot_diputado[col_out] = ventana.groupby('diputado')['es_afirmativo'].mean()

# Bloque/provincia actuales = los del voto mas reciente de cada diputado
ultimo_voto = (
    df_votos_raw.sort_values('fecha_votacion')
    .groupby('diputado')
    .tail(1)
    .set_index('diputado')
)
snapshot_diputado['bloque'] = ultimo_voto['bloque']
snapshot_diputado['provincia'] = ultimo_voto['provincia']

# Encoding con el MISMO encoder ya ajustado en STG_5 (no se reajusta)
enc = joblib.load(DATA_DIR / "encoder_bloque_provincia.joblib")
snapshot_diputado[['bloque_enc', 'provincia_enc']] = enc.transform(
    snapshot_diputado[['bloque', 'provincia']]
)

snapshot_diputado = snapshot_diputado.reset_index()
snapshot_diputado.to_csv(DATA_DIR / "snapshot_diputado.csv", index=False)

print(f"Diputados en el snapshot: {len(snapshot_diputado)}")
print()
print("NaN por columna (esperado: puede haber NaN en las tasas por ventana si el diputado")
print("no tiene votos en ese periodo; se resuelven con la cascada de relleno en la API):")
print(snapshot_diputado.isna().sum().to_string())
print()
print(f"Guardado: {DATA_DIR / 'snapshot_diputado.csv'}")
print()
print("T5 PASA.")

Diputados en el snapshot: 259

NaN por columna (esperado: puede haber NaN en las tasas por ventana si el diputado
no tiene votos en ese periodo; se resuelven con la cascada de relleno en la API):
diputado                       0
tasa_afirmativo_hist           0
tasa_alineacion_bloque_hist    0
n_votos_hist                   0
tasa_afirmativo_desde_2023     2
tasa_afirmativo_2026           3
bloque                         0
provincia                      0
bloque_enc                     0
provincia_enc                  0

Guardado: ..\data\snapshot_diputado.csv

T5 PASA.


## T6 — Snapshot de features del diputado por tema (acumulado total)

Mismo criterio que T5, pero desagregado por `tema_id`: para cada combinacion
(diputado, tema) se calcula la tasa de AFIRMATIVO usando **todos** los votos conocidos de
ese diputado en ese tema (sin `.shift`, mismo argumento que en T5: no hay voto "actual" que
excluir). Reusa `df_votos_raw` ya preparado en T5 y le suma el `tema_id` de cada titulo
(el mismo merge que hace `STG_5`).

In [6]:
# T6 - Snapshot de features del diputado por tema (acumulado total hasta hoy)
df_temas_map = pd.read_csv(DATA_DIR / "df_features_titulo.csv")[["titulo_base", "tema_id"]]

df_votos_tema = df_votos_raw.merge(df_temas_map, on="titulo_base", how="left")
assert df_votos_tema["tema_id"].isna().sum() == 0, "Quedaron titulos sin tema_id tras el merge"

snapshot_diputado_tema = (
    df_votos_tema.groupby(["diputado", "tema_id"])["es_afirmativo"]
    .mean()
    .reset_index()
    .rename(columns={"es_afirmativo": "tasa_afirmativo_tema_hist"})
)

snapshot_diputado_tema.to_csv(DATA_DIR / "snapshot_diputado_tema.csv", index=False)

combinaciones_posibles = snapshot_diputado["diputado"].nunique() * 20
print(f"Combinaciones diputado-tema con historial: {len(snapshot_diputado_tema):,} de {combinaciones_posibles:,} posibles")
print(f"NaN en tasa_afirmativo_tema_hist: {snapshot_diputado_tema['tasa_afirmativo_tema_hist'].isna().sum()}")
print()
print("Nota: las combinaciones diputado-tema SIN fila en esta tabla significan que ese diputado")
print("nunca voto sobre ese tema. En la API se resuelven con la cascada de relleno (cae a")
print("tasa_afirmativo_hist del diputado), igual que en STG_5.")
print()
print(f"Guardado: {DATA_DIR / 'snapshot_diputado_tema.csv'}")
print()
print("T6 PASA.")

Combinaciones diputado-tema con historial: 2,541 de 5,180 posibles
NaN en tasa_afirmativo_tema_hist: 0

Nota: las combinaciones diputado-tema SIN fila en esta tabla significan que ese diputado
nunca voto sobre ese tema. En la API se resuelven con la cascada de relleno (cae a
tasa_afirmativo_hist del diputado), igual que en STG_5.

Guardado: ..\data\snapshot_diputado_tema.csv

T6 PASA.


## T7 — Snapshot del historial del bloque por tema (acumulado total)

Mismo criterio que T6, pero a nivel de `bloque` en vez de diputado: para cada combinacion
(bloque, tema) se calcula la tasa de AFIRMATIVO usando todos los votos conocidos de
integrantes de ese bloque en ese tema. Reusa `df_votos_tema` (ya tiene `tema_id` mergeado
en T6).

In [7]:
# T7 - Snapshot del historial del bloque por tema (acumulado total hasta hoy)
snapshot_bloque_tema = (
    df_votos_tema.groupby(["bloque", "tema_id"])["es_afirmativo"]
    .mean()
    .reset_index()
    .rename(columns={"es_afirmativo": "tasa_afirmativo_bloque_tema"})
)

snapshot_bloque_tema.to_csv(DATA_DIR / "snapshot_bloque_tema.csv", index=False)

combinaciones_posibles = df_votos_tema["bloque"].nunique() * 20
print(f"Combinaciones bloque-tema con historial: {len(snapshot_bloque_tema):,} de {combinaciones_posibles:,} posibles")
print(f"NaN en tasa_afirmativo_bloque_tema: {snapshot_bloque_tema['tasa_afirmativo_bloque_tema'].isna().sum()}")
print()
print("Nota: las combinaciones bloque-tema sin historial se resuelven en la API con la misma")
print("cascada de relleno que STG_5 (cae a tasa_afirmativo_hist del diputado).")
print()
print(f"Guardado: {DATA_DIR / 'snapshot_bloque_tema.csv'}")
print()
print("T7 PASA.")

Combinaciones bloque-tema con historial: 571 de 1,100 posibles
NaN en tasa_afirmativo_bloque_tema: 0

Nota: las combinaciones bloque-tema sin historial se resuelven en la API con la misma
cascada de relleno que STG_5 (cae a tasa_afirmativo_hist del diputado).

Guardado: ..\data\snapshot_bloque_tema.csv

T7 PASA.


## T8 — Resumen: por qué esta notebook cumple la Constitución

Cada paso ya trae su propia explicación en la celda correspondiente. Este resumen las junta
para que el equipo pueda defenderlo de un vistazo, principio por principio.

**Principio 2 (validación temporal) y Principio 4 (F1-macro)** — Esta notebook **no evalúa
un modelo nuevo**. No se corre ningún `TimeSeriesSplit` ni se calcula ningún F1-macro que se
vaya a citar como métrica de desempeño. El único número de tipo F1-macro que aparece (T3,
0.989) es un chequeo interno de que la serialización no rompió nada — se documenta
explícitamente que **no reemplaza** al 0.453 de `STG_6`, que sigue siendo la métrica oficial
del proyecto (medida con un modelo que nunca vio el holdout).

**Principio 3 (cero data leakage)** — Los tres puntos donde podría colarse fuga de
información y por qué no ocurre:
- *Refit del LGBM sobre train+holdout (T2)*: leakage es usar información del futuro para
  **evaluar** un modelo. Acá no hay ninguna evaluación nueva — el holdout ya cumplió su
  función en `STG_6`/`STG_7`. Reentrenar con todo el historial para producción es una
  práctica estándar, no una trampa metodológica.
- *KMeans (T4)*: se ajusta una sola vez, de forma determinista, sobre los mismos 1.022
  títulos ya usados en `STG_4` — no ve ningún título nuevo. Para un título nuevo, la API
  usa siempre `.predict()`, nunca `.fit()`, así que nunca "aprende" del título que está
  prediciendo.
- *Snapshots (T5, T6, T7)*: se calculan como acumulado de **toda** la historia conocida,
  sin `.shift(1)`. Esto es correcto — no es lo mismo que el leakage de `STG_5` — porque acá
  no hay ningún "voto actual" que debería excluirse: se está prediciendo un voto
  **hipotético, futuro**, así que toda la historia real hasta hoy es información legítima
  y disponible.

**Principio 5 (reproducibilidad)** — `random_state=42` fijo en el LGBM (T2) y en el KMeans
(T4). Los artefactos se guardan con nombre propio, sin sobrescribir ningún dato crudo:

| Artefacto | Contenido |
|---|---|
| `data/modelo_lgbm.joblib` | Modelo LGBM ganador, reentrenado sobre train+holdout |
| `data/kmeans_temas.joblib` | KMeans de temas (K=20), para asignar tema a un título nuevo |
| `data/mapa_temas.json` | `tema_id → tema_label` (20 temas, incluye `sin_clasificar`) |
| `data/snapshot_diputado.csv` | Tasas históricas acumuladas por diputado (259 diputados) |
| `data/snapshot_diputado_tema.csv` | Tasa de AFIRMATIVO por (diputado, tema) |
| `data/snapshot_bloque_tema.csv` | Tasa de AFIRMATIVO por (bloque, tema) |

**Principio 7 (simple antes que sofisticado)** — No se agregó ninguna libreria nueva ni
ningún modelo nuevo: se reutiliza exactamente lo ya validado en la spec 009, solo se lo deja
guardado en disco.